Phase 3 -- Build the patient-similarity k-NN graph.

Reads the encoded feature matrix from Phase 2 and constructs a graph
where each patient is a node and edges connect the k most similar
patients (Euclidean distance over the 42-dim encoded feature vector).

Outputs an edge list (edge_index format, ready for PyTorch Geometric
in Phase 4) plus diagnostic plots/stats to validate the graph is
actually useful before we build a GNN on top of it.


In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import networkx as nx
import matplotlib.pyplot as plt

In [2]:
X_PATH = "Data/X_features.csv"
Y_PATH = "Data/y_labels.csv"
OUT_DIR = "Output"

In [3]:
X = pd.read_csv(X_PATH).values  # shape (704, 42)
y = pd.read_csv(Y_PATH).values.ravel()  # shape (704,)
n_nodes = X.shape[0]
print(f"Loaded feature matrix: {X.shape}, labels: {y.shape}")

Loaded feature matrix: (704, 42), labels: (704,)



STEP 1-2: k sensitivity analysis
Before committing to one k, let's see how graph structure changes
across a few candidate values. This is the empirical justification
for whatever k we finally pick -- much stronger than "we chose k=10
because it's common."


In [4]:
def build_knn_edges(X, k):
    """
    Returns a symmetrized, self-loop-free edge list for a k-NN graph.
    """
    # n_neighbors=k+1 because the point itself is (almost always) its
    # own nearest neighbor at distance 0 -- we fetch one extra and
    # explicitly filter out self, rather than assuming it's always at
    # rank 0. This matters because this dataset has a handful of exact
    # duplicate rows (identical encoded feature vectors for different
    # patients); when a duplicate exists, sklearn's distance-0 tie
    # between "self" and "the duplicate" is NOT guaranteed to place
    # self at rank 0, which caused real self-loops to slip through
    # when we naively skipped index 0.
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(X)
    distances, indices = nbrs.kneighbors(X)
 
    edges = set()
    for node_idx in range(X.shape[0]):
        neighbor_row = indices[node_idx]
        # explicitly drop self wherever it appears, then take the
        # first k of what remains
        neighbors_no_self = [n for n in neighbor_row if n != node_idx][:k]
        for neighbor_idx in neighbors_no_self:
            # Union symmetrization: add edge regardless of direction.
            # (Alternative: "mutual" k-NN only keeps edges where BOTH
            # nodes list each other as neighbors -- stricter, sparser,
            # sometimes fragments the graph. Union is the safer default
            # for a first pass.)
            edge = tuple(sorted((node_idx, int(neighbor_idx))))
            edges.add(edge)
    return edges
 
print("\n" + "=" * 60)
print("K SENSITIVITY ANALYSIS")
print("=" * 60)
for k_test in [3, 5, 10, 15, 20]:
    edges_test = build_knn_edges(X, k_test)
    g_test = nx.Graph()
    g_test.add_nodes_from(range(n_nodes))
    g_test.add_edges_from(edges_test)
    n_components = nx.number_connected_components(g_test)
    isolates = list(nx.isolates(g_test))
    avg_degree = 2 * len(edges_test) / n_nodes
    print(f"k={k_test:3d} | edges={len(edges_test):4d} | avg_degree={avg_degree:5.2f} "
          f"| connected_components={n_components:3d} | isolated_nodes={len(isolates)}")


K SENSITIVITY ANALYSIS
k=  3 | edges=1585 | avg_degree= 4.50 | connected_components=  1 | isolated_nodes=0
k=  5 | edges=2575 | avg_degree= 7.32 | connected_components=  1 | isolated_nodes=0
k= 10 | edges=5024 | avg_degree=14.27 | connected_components=  1 | isolated_nodes=0
k= 15 | edges=7425 | avg_degree=21.09 | connected_components=  1 | isolated_nodes=0
k= 20 | edges=9806 | avg_degree=27.86 | connected_components=  1 | isolated_nodes=0



#================================================================
STEP 3: Pick k based on the analysis above.
k=10 is a reasonable default: check the printed table above --
we want low isolated-node count and a small number of components
(ideally 1, meaning the whole graph is reachable) while keeping
average degree modest (not connecting everyone to everyone).
Change K_FINAL below if your printed table suggests a better value.
#================================================================

In [5]:
K_FINAL = 10
print(f"\nUsing K_FINAL = {K_FINAL} for the working graph")
 
edges = build_knn_edges(X, K_FINAL)
G = nx.Graph()
G.add_nodes_from(range(n_nodes))
G.add_edges_from(edges)


Using K_FINAL = 10 for the working graph


STEP 4: Structural sanity checks

In [6]:
print("\n" + "=" * 60)
print(f"GRAPH STRUCTURE (k={K_FINAL})")
print("=" * 60)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
degrees = [d for _, d in G.degree()]
print(f"Degree -- min: {min(degrees)}, max: {max(degrees)}, mean: {np.mean(degrees):.2f}")
print(f"Connected components: {nx.number_connected_components(G)}")
isolates = list(nx.isolates(G))
print(f"Isolated nodes: {len(isolates)}")
largest_cc = max(nx.connected_components(G), key=len)
print(f"Largest connected component size: {len(largest_cc)} / {n_nodes} "
      f"({100*len(largest_cc)/n_nodes:.1f}%)")

# Degree distribution plot
plt.figure(figsize=(6, 4))
plt.hist(degrees, bins=20, color="#4C72B0", edgecolor="white")
plt.xlabel("Node degree")
plt.ylabel("Count")
plt.title(f"Degree distribution (k={K_FINAL} similarity graph)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/degree_distribution.png", dpi=120)
plt.close()
print(f"\nSaved degree distribution plot to {OUT_DIR}/degree_distribution.png")



GRAPH STRUCTURE (k=10)
Nodes: 704
Edges: 5024
Degree -- min: 10, max: 33, mean: 14.27
Connected components: 1
Isolated nodes: 0
Largest connected component size: 704 / 704 (100.0%)

Saved degree distribution plot to Output/degree_distribution.png


STEP 5: Homophily check -- THE most important validation for this
project. GNN message passing assumes connected nodes tend to share
labels. If this ratio is close to 0.5 (random), the graph structure
carries little class signal and a GNN may not beat a tabular model.
If it's well above the base rate, the graph is doing real work.


In [7]:
same_label_edges = 0
for u, v in G.edges():
    if y[u] == y[v]:
        same_label_edges += 1
homophily_ratio = same_label_edges / G.number_of_edges()
 
# baseline: what fraction of edges would match by pure chance, given
# the class distribution?
p_asd = y.mean()
expected_random_homophily = p_asd**2 + (1 - p_asd)**2
 
print("\n" + "=" * 60)
print("HOMOPHILY CHECK (does the graph structure track the labels?)")
print("=" * 60)
print(f"Fraction of edges connecting same-label patients: {homophily_ratio:.3f}")
print(f"Expected fraction under random connections (baseline): {expected_random_homophily:.3f}")
if homophily_ratio > expected_random_homophily:
    print("--> Graph shows POSITIVE homophily: similar patients (by our "
          "features) tend to share the same diagnosis more than chance. "
          "Good signal that the GNN structure should help.")
else:
    print("--> WARNING: homophily is at/below the random baseline. The "
          "similarity graph may not carry useful class signal -- worth "
          "revisiting feature choice or k before trusting GNN gains.")


HOMOPHILY CHECK (does the graph structure track the labels?)
Fraction of edges connecting same-label patients: 0.885
Expected fraction under random connections (baseline): 0.607
--> Graph shows POSITIVE homophily: similar patients (by our features) tend to share the same diagnosis more than chance. Good signal that the GNN structure should help.


Visualize a random subgraph (full 704-node plot is unreadable)
colored by class label, so you can *see* whether same-colored nodes
cluster together -- a visual complement to the homophily number above.

In [8]:
np.random.seed(42)
sample_nodes = np.random.choice(n_nodes, size=80, replace=False)
subG = G.subgraph(sample_nodes)
 
plt.figure(figsize=(8, 8))
pos = nx.spring_layout(subG, seed=42, k=0.5)
node_colors = ["#E74C3C" if y[n] == 1 else "#3498DB" for n in subG.nodes()]
nx.draw_networkx(
    subG, pos, node_color=node_colors, node_size=120,
    with_labels=False, edge_color="#CCCCCC", width=0.8
)
plt.title("Patient similarity graph (80-node sample)\nRed = ASD positive, Blue = ASD negative")
plt.axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/graph_sample_visualization.png", dpi=120)
plt.close()
print(f"Saved sample graph visualization to {OUT_DIR}/graph_sample_visualization.png")

Saved sample graph visualization to Output/graph_sample_visualization.png


STEP 6: Save edge list in edge_index format for Phase 4 (PyTorch
Geometric expects a [2, num_edges] array; we save both directions
since PyG treats edge_index as directed under the hood -- an
"undirected" graph in PyG is represented by listing both (u,v) and (v,u)).

In [9]:
edge_list = list(G.edges())
src = [e[0] for e in edge_list] + [e[1] for e in edge_list]
dst = [e[1] for e in edge_list] + [e[0] for e in edge_list]
edge_index = np.array([src, dst])  # shape [2, 2*num_undirected_edges]
 
np.save(f"{OUT_DIR}/edge_index.npy", edge_index)
print(f"\nSaved edge_index (shape {edge_index.shape}) to {OUT_DIR}/edge_index.npy")
print("This is the PyG-ready directed edge list (both directions included).")


Saved edge_index (shape (2, 10048)) to Output/edge_index.npy
This is the PyG-ready directed edge list (both directions included).
